# Imports

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
import matplotlib.pyplot as plt
import helper_utils

## 1 Working with Multi-Feature Data

In this section, you will revisit the pizza delivery prediction problem using a dataset loaded from a `.csv` file. The dataset contains records from 100 previous deliveries.

### Dataset Features

* **`distance_miles`**
  Total delivery distance measured in miles.

* **`time_of_day_hours`**
  The time the order was dispatched for delivery, represented as a floating-point value using a 24-hour clock.

* **`is_weekend`**
  A binary indicator describing the type of day:

  * `1` → weekend
  * `0` → weekday

* **`delivery_time_minutes`**
  The target variable representing the total delivery duration in minutes.


### Business Constraints

The dataset follows these operational rules:

* Deliveries only occur between **8:00 AM (`8.0`)** and **8:00 PM (`20.0`)**
* Delivery distances are capped at **20 miles**


### 1.1 - Data ingestion and exploration

Load and understand your data. 

* Define the file path to your dataset file, `./data_with_features.csv`.
* Use Pandas library to load the dataset from the given file path as a DataFrame, `data_df`, a powerful structure for manipulating and analyzing data.
* Inspect the shape of your data, which will show as **100 rows** (representing 100 deliveries) and **4 columns**.

In [ ]:
# Load the dataset from the CSV file
file_path = './data_with_features.csv'
data_df = pd.read_csv(file_path)

# Print the shape of the DataFrame
print(f"Dataset Shape: {data_df.shape}\n")

In [ ]:
data_df.head()

Now that the dataset has been loaded, the next step is to visualize it in order to better understand how the input features relate to the target variable.

The helper function `plot_delivery_data` below generates a scatter plot that displays all four features simultaneously:

* The **x-axis** represents the delivery **distance**
* The **y-axis** represents the **delivery time**
* The **color** of each point corresponds to the **time of day**

  * lighter shades indicate earlier dispatch times
  * darker red shades indicate later dispatch times
* The **marker style** indicates whether the delivery occurred on a weekday or weekend

  * solid circles represent weekdays
  * hollow circles represent weekends

As you examine the visualization, look for trends or relationships between the features and the delivery time. Consider how factors such as distance, dispatch time, and weekends may affect delivery duration.


In [ ]:
helper_utils.plot_delivery_data(data_df)

### 1.2 — Feature Engineering

From the plot above, it is clear that some deliveries take longer than others even when the travel distance is similar. A likely explanation is increased traffic congestion during rush hours.

Rather than expecting the model to automatically learn this complex relationship, we can apply **feature engineering**. Feature engineering involves using domain knowledge to create new variables that make important patterns in the data more explicit for the model.

In this section, you will create a new feature that identifies whether a delivery was dispatched during a rush hour period.

> The new feature will take the value `1` if the delivery was dispatched during:
>
> - the **morning rush hour**: `08:00 – 10:00`
> - the **evening rush hour**: `16:00 – 19:00`
>
> on a **weekday**, and `0` otherwise.

You may notice that rush hour is only considered on weekdays. This reflects a common real-world traffic pattern: rush hours are primarily driven by weekday commuter activity. On weekends, traffic patterns tend to differ significantly, and the predictable commuter congestion observed during the workweek is usually absent. Therefore, restricting rush hour logic to weekdays is a realistic and practical assumption for this task.

Before applying this logic to the full dataset, it is good practice to first test the implementation on a small sample. Working with a smaller subset makes it easier to debug and validate the feature engineering process.

For this exercise:

- Define the first **5 rows** of `data_df` as a **PyTorch tensor**.
- A tensor is used here because the complete dataset will later be processed in tensor form as well.
- Using tensors from the beginning ensures that the feature engineering function developed here can be applied directly to the full dataset without modification.
- This initial tensor should contain all available information for each sample delivery.


In [ ]:
# Define the 5 rows of data as a single 2D tensor
sample_tensor = torch.tensor([
    # distance, time_of_day, is_weekend, delivery_time
    [1.60,      8.20,        0,          7.22],   # row 1
    [13.09,     16.80,       1,          32.41],  # row 2       
    [6.97,      8.02,        1,          17.47],  # row 3
    [10.66,     16.07,       0,          37.17],  # row 4
    [18.24,     13.47,       0,          38.36]   # row 5
], dtype=torch.float32)

* To create the rush hour feature, the calculation only requires two pieces of information:
    - the **time of day**
    - whether the delivery occurred on a **weekday**

* The remaining columns, such as distance and delivery duration, are not needed for this step.

* Use **tensor slicing** to extract only the relevant columns from the dataset:
    - `time_of_day_hours` → column index `1`
    - `is_weekend` → column index `2`

* This operation creates a smaller tensor containing only the features required for the rush hour calculation.

In [ ]:
# Use tensor slicing to separate out each column
# Slicing syntax is [:, column_index]
sample_hours = sample_tensor[:, 1]
sample_weekends = sample_tensor[:, 2]

print("--- Sliced Tensors ---")
print(f"Sample Hours:    {sample_hours}")
print(f"Sample Weekends: {sample_weekends}\n")

Now that the `sample_hours` and `sample_weekends` tensors have been prepared, you will use them to build the `rush_hour_feature` function.

### Create `rush_hour_feature`

Implement the `rush_hour_feature` function.

#### Your Task

##### 1. Define the individual conditions

Create the following boolean tensors:

- `is_morning_rush`
    - `True` where `hours_tensor` is:
        - greater than or equal to `8.0`
        - **AND**
        - less than `10.0`

- `is_evening_rush`
    - `True` where `hours_tensor` is:
        - greater than or equal to `16.0`
        - **AND**
        - less than `19.0`

- `is_weekday`
    - `True` where `weekends_tensor` is equal to `0`

##### 2. Combine the conditions

Create `is_rush_hour_mask` by combining the boolean tensors above.

The condition should evaluate to `True` only when:

- the delivery occurs on a **weekday**
- **AND**
- the time falls within either:
    - the morning rush window
    - **OR**
    - the evening rush window


**Hint:**  
PyTorch tensors support element-wise comparison and logical operations directly using operators such as:

- `>=`
- `<`
- `==`
- `&` for logical AND
- `|` for logical OR

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if needed)</font></b></summary>

Build each boolean mask one step at a time.

### For `is_morning_rush`

This condition requires two comparisons on `hours_tensor` combined with a logical **AND** (`&`).

One part of the condition is already shown below:

```python
(hours_tensor >= 8.0)

In [ ]:
# FUNCTION rush_hour_feature

def rush_hour_feature(hours_tensor, weekends_tensor):
    """
    Engineers a new binary feature indicating if a delivery is in a weekday rush hour.

    Args:
        hours_tensor (torch.Tensor): A tensor of delivery times of day.
        weekends_tensor (torch.Tensor): A tensor indicating if a delivery is on a weekend.

    Returns:
        torch.Tensor: A tensor of 0s and 1s indicating weekday rush hour.
    """

    ### START CODE HERE: replace None ###
    
    # Define rush hour and weekday conditions
    is_morning_rush = None
    is_evening_rush = None
    is_weekday = weekends_tensor==None

    # Combine the conditions to create the final rush hour mask
    is_rush_hour_mask =  None

    ### END CODE HERE ###

    # Convert the boolean mask to a float tensor to use as a numerical feature
    return is_rush_hour_mask.float()

In [ ]:
rush_hour_for_sample = rush_hour_feature(sample_hours, sample_weekends)

print(f"Sample Hours:     {sample_hours.numpy()}")
print(f"Sample Weekends:  {sample_weekends.numpy()}")
print(f"Is Rush Hour?:    {rush_hour_for_sample.numpy()}")

#### Expected Output

```
Sample Hours:     [ 8.2  16.8   8.02 16.07 13.47]
Sample Weekends:  [0. 1. 1. 0. 0.]
Is Rush Hour?:    [1. 0. 0. 1. 0.]
```

### 1.3 — Building the Data Preparation Pipeline

Now that the feature engineering function has been implemented, the next step is to integrate it into a complete data preparation pipeline.

The objective is to create a single function that:

- accepts the raw pandas DataFrame as input
- performs all required preprocessing steps
- returns the final `features` and `targets` tensors used for model training

This pipeline will carry out several important transformations:

- generate the engineered rush hour feature using `rush_hour_feature()`
- normalize the `distance_miles` and `time_of_day_hours` columns so they are on a comparable scale
- perform the required tensor manipulations to correctly structure the dataset for a neural network

At the end of this process, you will obtain:

- a single `features` tensor
- a single `targets` tensor

both formatted correctly for training.

###  Build the `prepare_data` function


Implement the core tensor manipulation steps inside the `prepare_data` function.

The normalization logic and final feature concatenation are already provided.


### 1. Convert the DataFrame to a Tensor

Convert `all_values` (extracted from the pandas DataFrame) into a PyTorch tensor named `full_tensor`.

Requirements:

- use `torch.tensor()`
- set the data type to `torch.float32`

### 2. Slice the Tensor into Individual Variables

Use tensor slicing to separate `full_tensor` into individual 1D tensors:

- `raw_distances` → column index `0`
- `raw_hours` → column index `1`
- `raw_weekends` → column index `2`
- `raw_targets` → column index `3`

### 3. Create the Engineered Feature

Use the `rush_hour_feature()` function created earlier.

Pass the following tensors to the function:

- `raw_hours`
- `raw_weekends`

Store the result in:

```python id="lq7g7g"
is_rush_hour_feature

In [ ]:
# Function prepare_data

def prepare_data(df):
    """
    Converts a pandas DataFrame into prepared PyTorch tensors for modeling.

    Args:
        df (pd.DataFrame): A pandas DataFrame containing the raw delivery data.

    Returns:
        prepared_features (torch.Tensor): The final 2D feature tensor for the model.
        prepared_targets (torch.Tensor): The final 2D target tensor.
        results_dict (dict): A dictionary of intermediate tensors for testing purposes.
    """

    # Extract the data from the DataFrame as a NumPy array
    # (There's no direct torch.from_dataframe(), so we use .values to get a NumPy array first)
    all_values = df.values

    ### CODE HERE ###

    # Convert all the values from the DataFrame into a single PyTorch tensor
    full_tensor = None

    # Use tensor slicing to separate out each raw column
    raw_distances = None
    raw_hours = None
    raw_weekends = None
    raw_targets = None

    # Call your rush_hour_feature() function to engineer the new feature
    is_rush_hour_feature = rush_hour_feature(raw_hours, raw_weekends)

    # Use the .unsqueeze(1) method to reshape the four 1D feature tensors into 2D column vectors
    distances_col = None
    hours_col = None
    weekends_col = None
    rush_hour_col = None

    ### END CODE HERE ###

    # Normalize the continuous feature columns (distance and time)
    dist_mean, dist_std = distances_col.mean(), distances_col.std()
    hours_mean, hours_std = hours_col.mean(), hours_col.std()
 
    distances_norm = (distances_col - dist_mean) / dist_std
    hours_norm = (hours_col - hours_mean) / hours_std

    # Combine all prepared 2D features into a single tensor
    prepared_features = torch.cat([
        distances_norm,
        hours_norm,
        weekends_col,
        rush_hour_col
    ], dim=1) # dim=1 concatenates them column-wise, stacking features side by side

    # Prepare targets by ensuring they are the correct shape
    prepared_targets = raw_targets.unsqueeze(1)
    
    # Dictionary for Testing Purposes
    results_dict = {
        'full_tensor': full_tensor,
        'raw_distances': raw_distances,
        'raw_hours': raw_hours,
        'raw_weekends': raw_weekends,
        'raw_targets': raw_targets,
        'distances_col': distances_col,
        'hours_col': hours_col,
        'weekends_col': weekends_col,
        'rush_hour_col': rush_hour_col
    }
    

    return prepared_features, prepared_targets, results_dict

In [ ]:
# Create a small test DataFrame with the first 5 entries
test_df = data_df.head(5).copy()

# Print the "Before" state as a raw tensor
raw_test_tensor = torch.tensor(test_df.values, dtype=torch.float32)
print("--- Raw Tensor (Before Preparation) ---\n")
print(f"Shape: {raw_test_tensor.shape}")
print("Values:\n", raw_test_tensor)
print("\n" + "="*50 + "\n")

# Run the function to get the prepared "after" tensors
test_features, test_targets, _ = prepare_data(test_df)

# Print the "After" state
print("--- Prepared Tensors (After Preparation) ---")
print("\n--- Prepared Features ---\n")
print(f"Shape: {test_features.shape}")
print("Values:\n", test_features)

print("\n--- Prepared Targets ---")
print(f"Shape: {test_targets.shape}")
print("Values:\n", test_targets)

#### Expected Output

```
--- Prepared Tensors (After Preparation) ---

--- Prepared Features ---

Shape: torch.Size([5, 4])
Values:
 tensor([[-1.3562, -1.0254,  0.0000,  1.0000],
        [ 0.4745,  1.0197,  1.0000,  0.0000],
        [-0.5006, -1.0682,  1.0000,  0.0000],
        [ 0.0873,  0.8461,  0.0000,  1.0000],
        [ 1.2951,  0.2278,  0.0000,  0.0000]])

--- Prepared Targets ---
Shape: torch.Size([5, 1])
Values:
 tensor([[ 7.2200],
        [32.4100],
        [17.4700],
        [37.1700],
        [38.3600]])
```

We have now prepared:

* **A "features" tensor**: This contains the four columns of input data (distance, time of day, weekend flag, rush hour flag) the model will learn from. Notice how the first two columns have been normalized, and the fourth column is your newly engineered `is_rush_hour` feature.

* **A "targets" tensor**: This contains only the final `delivery_time_minutes`, separated from the input features. This is the value your model will learn to predict.

We have built and verified our data preparation pipeline on a small sample sampple. Next, we will run it on the entire dataset to prepare all the data for training.

In [ ]:
# Process the entire DataFrame to get the final feature and target tensors.
# the underscore _ is used as a placeholder variable to indicate that the third returned value from the prepare_data(data_df) function 
# is being intentionally ignored.

features, targets, _ = prepare_data(data_df)

### 1.4 — Visualizing the Prepared Data

Now that we built the data preparation pipeline, we need to plot the resulting data.
The  plots **Delivery Time** against **Normalized Distance**  provide a view of how the target variable relates to a scaled input feature.

The points are categorized into four groups based on:

- day type (weekday vs. weekend)
- rush hour status (rush hour vs. non-rush hour)

This combined encoding allows the plot to reflect both temporal structure and traffic conditions simultaneously.


**“Weekend (Rush Hour)”**  does not appear in the data since rush hour labeling is applied to weekdays only, as defined in the pipeline.

In [ ]:
helper_utils.plot_final_data(features, targets)

## 2 - Modelling. Building the Neural Network

With your data pipeline complete, we will build a neural network with two hidden layers to capture the complex relationships between all your input features.

### Build the init_model function

Next, we implement the `init_model` function, to define the model architecture, the optimizer, and the loss function.

**Your Task:**

* **Define the Model Architecture**:
    * Define a `model` using `nn.Sequential`.
    * The model **should only** have three `nn.Linear` layers, each followed by a `nn.ReLU()` activation function, except for the last one.
        * **Input Layer**: An `nn.Linear` layer that accepts **4 input features** and outputs **64 features**.
        * **Hidden Layer**: An `nn.Linear` layer that takes the **64 features** from the previous layer and outputs **32 features**.
        * **Output Layer**: A final `nn.Linear` layer that takes the **32 features** from the hidden layer and produces a **single output** value.
>        
* **Define the Optimizer**:
    * Define the `optimizer` as **Stochastic Gradient Descent (SGD)**. You need to pass it the model's parameters (`model.parameters()`) and set the learning rate (`lr`) to `0.01`.
>
* **Define the Loss Function**:
    * Define the `loss_function` as **Mean Squared Error (MSE)**.
 
<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
**For the Model:**
* Remember to list your layers inside the `nn.Sequential()` constructor, separated by commas.
* The `nn.Linear()` layer takes two main arguments: `in_features` and `out_features`. Ensure the `in_features` of one layer matches the `out_features` of the one before it.
* The correct order of layers is: **Input Layer -> ReLU -> Hidden Layer -> ReLU -> Output Layer.**

**For the Optimizer:**
* You will use `optim.SGD`. Its first argument is the model's parameters, which you can get with `model.parameters()`.
* The second argument you need to provide is the learning rate, `lr=0.01`.

**For the Loss Function:**
* You will use `nn.MSELoss`. Since it is a class, you need to create an instance of it by calling it with parentheses: `nn.MSELoss()`.


In [ ]:
# Function init_model

n_input_features = features.size(1)
n_output_features = targets.size(1)

def init_model():
    """
    Initializes the neural network model, optimizer, and loss function.

    Returns:
        model (nn.Sequential): The initialized PyTorch sequential model.
        optimizer (torch.optim.Optimizer): The initialized optimizer for training.
        loss_function: The initialized loss function.
    """

    # Set the random seed for reproducibility of results (DON'T MANIPULATE IT)
    torch.manual_seed(41)

    ### REPLACE NONE ###

    # Define the model architecture using nn.Sequential
    model =  nn.Sequential(
        # Input layer (Linear): input features  n_input_features, 64 output features
        None
        # First ReLU activation function
        None
        # Hidden layer (Linear): 64 inputs, 32 outputs
        None
        # Second ReLU activation function
        None
        # Output layer (Linear): 32 inputs, 1 output (n_output_features)
        None
    ) 
    
    # Define the optimizer (Stochastic Gradient Descent)
    optimizer = None

    # Define the loss function (Mean Squared Error for regression)
    loss_function = None

    ### END CODE HERE ###

    return model, optimizer, loss_function

In [ ]:
model, optimizer, loss_function = init_model()

print(f"{'='*30}\nInitialized Model Architecture\n{'='*30}\n{model}")
print(f"\n{'='*30}\nOptimizer\n{'='*30}\n{optimizer}")
print(f"\n{'='*30}\nLoss Function\n{'='*30}\n{loss_function}")

#### Expected Output:

```
==============================
Initialized Model Architecture
==============================
Sequential(
  (0): Linear(in_features=4, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)

==============================
Optimizer
==============================
SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

==============================
Loss Function
==============================
MSELoss()
```

## 3 - Training the Model

With your data prepared and your model architecture defined, it's time for the most important stage: **training**. 

### Exercise 4 - train_model

Implement the complete training loop inside the `train_model` function.

**Your Task:**

* **Initialize your model and tools:**
    * Start by calling the `init_model()` function you built earlier to get your `model`, `optimizer`, and `loss_function`.
>    
* **Loop through the epochs:**
    * Create a `for` loop that iterates from 0 up to the number of `epochs` provided.
>    
* **Implement the training steps inside the loop:**
    * Perform these five steps **in order** on each iteration:
        * **Forward Pass**: Pass your `features` tensor into the `model` to get its predictions.
        * **Calculate Loss**: Use your `loss_function` to compare the model's predictions with the actual `targets`.
        * **Zero Gradients**: Zero the gradients on the `optimizer` from the previous iteration.
        * **Backward Pass**: Perform the backward pass on your `loss` to calculate the new gradients.
        * **Update Weights**: Take a step with the `optimizer` to update the model's parameters.


<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
**For Initialization:**
* The `init_model()` function returns three values. You can unpack them directly into your three variables: `model, optimizer, loss_function = init_model()`.

**For the Forward Pass:**
* To get predictions, you can call your `model` object like a "function", passing the `features` as the "argument", `function(argument)`.

**For Calculating Loss:**
* The loss function also works like a function. It takes two arguments, your "predictions" and "actual targets".

**For the Gradient Steps:**
* The three gradient-related steps (`.zero_grad()`, `.backward()`, and `.step()`) are all methods that need to be called with parentheses, for example, `optimizer.zero_grad()`.

In [ ]:
# Function train_model

def train_model(features, targets, epochs, verbose=True):
    """
    Trains the model using the provided data for a number of epochs.
    
    Args:
        features (torch.Tensor): The input features for training.
        targets (torch.Tensor): The target values for training.
        epochs (int): The number of training epochs.
        verbose (bool): If True, prints training progress. Defaults to True.
        
    Returns:
        model (nn.Sequential): The trained model.
        losses (list): A list of loss values recorded every 5000 epochs.
    """
    
    # Initialize a list to store the loss
    losses = []
    
    ### REPLACE NONE ###
    
    # Initialize the model, optimizer, and loss function using `init_model`
    model, optimizer, loss_function = init_model()

    # Loop through the specified number of epochs
    for epoch in range(epochs):
        
        # Forward pass: Make predictions
        outputs = None

        # Calculate the loss
        loss = None

        # Zero the gradients
        None

        # Backward pass: Compute gradients
        None

        # Update the model's parameters
        None
    
    ### END CODE HERE ### 

        # Every 5000 epochs, record the loss and print the progress
        if (epoch + 1) % 5000 == 0:
            losses.append(loss.item())
            if verbose:
                print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")
    
    return model, losses

In [ ]:
test_model, loss = train_model(features, targets, 10000)

#### Expected Output (approximately):

```
Epoch [5000/10000], Loss: 3.0901
Epoch [10000/10000], Loss: 1.6064
```

It's time to put your `train_model` function to work. Run the complete training on the `features` and `targets`. You will train the model for `30,000` epochs (more than the test run to ensure full convergence on the complete dataset), which gives it ample opportunity to learn the patterns in the data.

In [ ]:
# Training loop
model, loss = train_model(features, targets, 30000)

## 4 - Evaluating Model Performance

Now that your model is trained, it's time to evaluate its performance. A simple yet powerful way to do this for a regression task is to plot the model's predictions against the actual target values.

* First, use your trained `model` to get predictions for the entire dataset.
* Then, create a scatter plot, **Actual Delivery Times (x-axis) vs. Predicted Delivery Times (y-axis)**.
* If the model is accurate, the points on the plot should form a tight cluster along a straight diagonal line.
    * The closer the points are to this line, the better your model's predictions are.
    

In [ ]:
# Disable gradient calculation for efficient predictions
with torch.no_grad():
    # Perform a forward pass to get model predictions
    predicted_outputs = model(features)

# Plot predictions vs. actual targets to evaluate performance
helper_utils.plot_model_predictions(predicted_outputs, targets)

## 5 - Making a New Prediction

With a well-trained and evaluated model, you've reached the final and most practical stage: **prediction**. It's time to use your model to make a prediction on new, unseen data. 

* Define a new delivery scenario by setting the distance, time of day, and whether it's a weekend.

**Note on Business Rules:**
Remember the constraints of the delivery service when setting your values:
* **Distance**: Must be less than or equal to `20` miles.
* **Time**: Must be between `8.0` (8:00 AM) and `20.0` (8:00 PM).
* **Weekend**: Can be set using `True`/`False` or `1`/`0`.

In [ ]:
#  Set new values below

# Change the values below to get an estimate for a different delivery
# Set distance for the delivery in miles
distance_miles = 8 
# Set time of day in 24-hour format (e.g., 9.5 for 9:30 AM)
time_of_day_hours = 12
# Use True/False or 1/0 to indicate if it's a weekend
is_weekend = False

# Convert the raw inputs into a 2D tensor for the model
raw_input_tensor = torch.tensor([[distance_miles, time_of_day_hours, is_weekend]], dtype=torch.float32)

Now, you'll pass your trained `model`, the original `data_df`, your `raw_input_tensor`, and the `rush_hour_feature` function to the helper function. This will process your new inputs and use the model to generate the estimated delivery time.

In [ ]:
helper_utils.prediction(model, data_df, raw_input_tensor, rush_hour_feature)

# Implementing Mini-Batch Gradient Descent using DataLoaders


We now transition from full-batch training on raw tensors to a mini-batch training workflow using PyTorch’s data utilities.

We represent the dataset using `TensorDataset`, a lightweight PyTorch abstraction that binds input feature tensors and target tensors into a single indexed dataset. This enables consistent paired access to samples during training.

We then construct a `DataLoader`, which provides an efficient interface for iterating over the dataset in mini-batches. It supports key training utilities such as batching, shuffling, and parallel data loading, making it the standard mechanism for feeding data into PyTorch training loops.

The model initialization is adapted to this structure so that it remains compatible with batched inputs and correctly reflects the dataset-defined input and output dimensionality.

The training loop is refactored to operate over mini-batches. For each batch, the model performs a forward pass, computes the loss, executes backpropagation, and updates parameters via the optimizer. This replaces the previous full-dataset computation with stochastic gradient-based updates.

Finally, the training invocation is updated to accept the `DataLoader`, aligning the entire pipeline with standard PyTorch training practices.

Overall, this implementation reflects the canonical PyTorch workflow for supervised learning, enabling scalable and efficient optimization in practical deep learning systems.


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
train_dataset = TensorDataset(features, targets)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

In [ ]:
# Determine input and output dimensions from the dataset tensors
n_input_features = features.size(1)
n_output_features = targets.size(1)


def init_model(train_loader):
    """
    Initializes the neural network model, optimizer, and loss function
    for the pizza delivery prediction problem.

    Args:
        train_loader (DataLoader): PyTorch DataLoader containing
                                   pizza delivery batches.

    Returns:
        model (nn.Sequential): Initialized neural network.
        optimizer (torch.optim.Optimizer): Optimizer for training.
        loss_function: Loss function used during training.
    """

    # Set the random seed for reproducibility (DO NOT CHANGE)
    torch.manual_seed(41)

    ### REPLAVE NONE

    # Define neural network architecture
    model = nn.Sequential(
        # Input layer
        None,

        # Activation
        None,

        # Hidden layer
        None,

        # Activation
        None,

        # Output layer
        None
    )

    # Optimizer
    optimizer = None

    # Loss function for regression
    loss_function = None

    ### END YOUR CODE HERE ££££

    # Optional sanity check using one batch from the DataLoader
    sample_features, sample_targets = next(iter(train_loader))

    print("Batch feature shape:", sample_features.shape)
    print("Batch target shape:", sample_targets.shape)

    return model, optimizer, loss_function

In [ ]:
model, optimizer, loss_function = init_model(train_loader)

In [ ]:
model, optimizer, loss_function = init_model(train_loader)

In [ ]:
def train_model(train_loader, epochs, verbose=True):
    """
    Trains the model using mini-batches from a DataLoader.

    Args:
        train_loader (DataLoader): DataLoader providing training batches.
        epochs (int): Number of training epochs.
        verbose (bool): If True, prints training progress.

    Returns:
        model (nn.Sequential): The trained model.
        losses (list): Recorded loss values every 5000 epochs.
    """

    losses = []

    # Initialize model components
    model, optimizer, loss_function = init_model(train_loader)

    # Training loop over epochs
    for epoch in range(epochs):

        # Iterate over mini-batches
        for batch_features, batch_targets in train_loader:

            ### REPLACE NONE ######

            # Forward pass
            None

            # Compute loss
            loss = None

            # Backpropagation
            None
            None
            None

            ### END YOUR CODE HERE ####

        # Log progress every 5000 epochs
        if (epoch + 1) % 5000 == 0:
            losses.append(loss.item())
            if verbose:
                print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

    return model, losses

In [ ]:
test_model, loss = train_model(train_loader, epochs=10000)

In [ ]:
# Training loop
model, loss = train_model(train_loader, epochs = 30000)

In [ ]:
# Set model to evaluation mode
model.eval()

all_predictions = []
all_targets = []

# Disable gradient calculation for efficient inference
with torch.no_grad():

    # Iterate over DataLoader batches
    for batch_features, batch_targets in train_loader:

        # Forward pass
        batch_outputs = model(batch_features)

        # Store results
        all_predictions.append(batch_outputs)
        all_targets.append(batch_targets)

# Concatenate all batches
predicted_outputs = torch.cat(all_predictions, dim=0)
targets = torch.cat(all_targets, dim=0)

# Return to training mode (optional but good practice)
model.train()

# Plot predictions vs actual targets
helper_utils.plot_model_predictions(predicted_outputs, targets)

That's the end of this session